In [1]:
import chess
import torch
import numpy as np

In [5]:
planes = torch.zeros((12, 8, 8), dtype=torch.float32)
planes.shape

torch.Size([12, 8, 8])

In [7]:
planes.flatten().shape

torch.Size([768])

In [8]:
print(planes.shape)
print(planes.ndim)
print(planes.numel())

torch.Size([12, 8, 8])
3
768


In [9]:
planes[0, 6, 4] = 1

print(planes[0])

tensor([[0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 1., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0.]])


In [15]:
torch.zeros(5)

tensor([0., 0., 0., 0., 0.])

In [16]:
def encode_board(board: chess.Board) -> torch.Tensor:
    planes = torch.zeros((12, 8, 8), dtype=torch.float32)

    for square, piece in board.piece_map().items():
        plane_index = PIECE_TO_PLANE[(piece.color, piece.piece_type)]

        row = 7 - chess.square_rank(square)
        col = chess.square_file(square)

        planes[plane_index, row, col] = 1.0

    flat = planes.flatten()
    # [white_to_move, white_K, white_Q, black_K, black_Q]
    features = torch.empty(5, dtype=torch.float32)
    features[0] = float(board.turn == chess.WHITE)
    features[1] = float(board.has_kingside_castling_rights(chess.WHITE))
    features[2] = float(board.has_queenside_castling_rights(chess.WHITE))
    features[3] = float(board.has_kingside_castling_rights(chess.BLACK))
    features[4] = float(board.has_queenside_castling_rights(chess.BLACK))

    result = torch.cat((flat, features))
    return result

In [17]:
PIECE_TO_PLANE = {
    (chess.WHITE, chess.PAWN): 0,
    (chess.WHITE, chess.KNIGHT): 1,
    (chess.WHITE, chess.BISHOP): 2,
    (chess.WHITE, chess.ROOK): 3,
    (chess.WHITE, chess.QUEEN): 4,
    (chess.WHITE, chess.KING): 5,
    (chess.BLACK, chess.PAWN): 6,
    (chess.BLACK, chess.KNIGHT): 7,
    (chess.BLACK, chess.BISHOP): 8,
    (chess.BLACK, chess.ROOK): 9,
    (chess.BLACK, chess.QUEEN): 10,
    (chess.BLACK, chess.KING): 11,
}

In [18]:
from neurachess.neural.encoding import encode_board

board = chess.Board()
x = encode_board(board)

print("Shape:", x.shape)
print("Pieces:", x[:768].sum().item())
print("Global features:", x[-5:])

Shape: torch.Size([773])
Pieces: 32.0
Global features: tensor([1., 1., 1., 1., 1.])


In [19]:
board.push_uci("e2e4")
x = encode_board(board)

print(x.shape)
print(x[-5:])

torch.Size([773])
tensor([0., 1., 1., 1., 1.])
